# 第9章 文脈を広げる：埋め込みと多層パーセプトロン

直前8文字を見る、本物のニューラル言語モデル

教科書のページ：https://sakunokyumei.github.io/llm-research-textbook/09-mlp.html

上のセルから順に、再生ボタン（▶）か `Shift + Enter` で実行します。説明は教科書のページにあります。


## 準備


In [ ]:
import re, math, urllib.request
import torch
import torch.nn as nn
import torch.nn.functional as F

def load_soseki(file_id="773_ruby_5968"):
    url = ("https://raw.githubusercontent.com/aozorahack/aozorabunko_text/master/"
           f"cards/000148/files/{file_id}/{file_id}.txt")
    text = urllib.request.urlopen(url).read().decode("cp932")
    text = text.split("-------------------------------------------------------")[2]
    text = text.split("底本：")[0]
    text = re.sub(r"《.*?》", "", text)
    text = re.sub(r"※?［＃.*?］", "", text)
    text = text.replace("｜", "").replace("\r\n", "\n")
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

text = load_soseki()
chars = sorted(set(text))
V = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
data = torch.tensor([stoi[c] for c in text])
n = int(len(data) * 0.9)
train_data, val_data = data[:n], data[n:]

## 8文字の窓


In [ ]:
T = 8   # 窓の大きさ（何文字前まで見るか）

def get_batch(data, batch_size, g):
    ix = torch.randint(0, len(data) - T, (batch_size,), generator=g)  # 窓の開始位置をランダムに
    x = torch.stack([data[i:i + T] for i in ix])     # 入力：8文字
    y = data[ix + T]                                 # 正解：その次の1文字
    return x, y

g = torch.Generator().manual_seed(0)
x, y = get_batch(train_data, 4, g)
print(x.shape, y.shape)
for i in range(4):
    print(repr("".join(itos[j] for j in x[i].tolist())), "→", repr(itos[y[i].item()]))

## 埋め込み：文字を「点」にする


In [ ]:
torch.manual_seed(0)
C = torch.randn(V, 2)          # 2061文字 × 2個の数（最初はでたらめ）
print("「先」の座標：", C[stoi["先"]])
print("「奥」の座標：", C[stoi["奥"]])
print(C[x].shape)              # 4問 × 8文字 × 2個の数

## 層を重ねる：多層パーセプトロン


In [ ]:
d, H = 32, 256    # 埋め込みの大きさ、隠れ層の大きさ
torch.manual_seed(0)
C = torch.randn(V, d)
W1 = torch.randn(T * d, H) * 0.1
b1 = torch.zeros(H)
W2 = torch.randn(H, V) * 0.1
b2 = torch.zeros(V)

emb = C[x]                       # [4, 8, 32]
flat = emb.view(4, T * d)        # [4, 256]：8文字ぶんを横一列に並べる
h = torch.tanh(flat @ W1 + b1)   # [4, 256]：隠れ層
logits = h @ W2 + b2             # [4, 2061]：次の文字の点数
print(emb.shape, flat.shape, h.shape, logits.shape)

## なぜ tanh を挟むのか：非線形性


In [ ]:
!pip install -q matplotlib-fontja

In [ ]:
import matplotlib.pyplot as plt
import matplotlib_fontja

z = torch.linspace(-4, 4, 200)
plt.figure(figsize=(6, 3.5))
plt.plot(z, torch.tanh(z), label="tanh")
plt.plot(z, torch.relu(z), label="ReLU")
plt.plot(z, F.gelu(z), label="GELU", linestyle="--")
plt.axhline(0, color="gray", lw=0.5); plt.axvline(0, color="gray", lw=0.5)
plt.ylim(-1.5, 3)
plt.legend()
plt.title("いろいろな活性化関数")
plt.show()

## 部品をまとめる：クラスで書く


In [ ]:
class MLP(nn.Module):
    def __init__(self, V, T, d=32, H=256):
        super().__init__()
        self.emb = nn.Embedding(V, d)     # 埋め込みの表（C）
        self.fc1 = nn.Linear(T * d, H)    # 隠れ層（W1, b1）
        self.fc2 = nn.Linear(H, V)        # 出力層（W2, b2）

    def forward(self, x):
        e = self.emb(x)                          # [B, T, d]
        h = torch.tanh(self.fc1(e.view(len(x), -1)))  # [B, H]
        return self.fc2(h)                       # [B, V]

torch.manual_seed(0)
model = MLP(V, T)
print(model)
print("パラメータの数：", sum(p.numel() for p in model.parameters()))

## 学習させる


In [ ]:
@torch.no_grad()
def val_loss(model):
    X = torch.stack([val_data[i:i + T] for i in range(len(val_data) - T)])
    Y = val_data[T:]
    return F.cross_entropy(model(X), Y).item()

torch.manual_seed(0)
g = torch.Generator().manual_seed(0)
model = MLP(V, T)
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

for step in range(10001):
    x, y = get_batch(train_data, 256, g)
    loss = F.cross_entropy(model(x), y)
    optimizer.zero_grad()     # 勾配を0に
    loss.backward()           # 勾配を計算
    optimizer.step()          # パラメータを更新
    if step % 2000 == 0:
        print(f"ステップ {step:5d}：訓練 {loss.item():.3f}  検証 {val_loss(model):.3f}")

## 書かせてみる


In [ ]:
@torch.no_grad()
def generate(model, start, n=100, seed=0):
    g = torch.Generator().manual_seed(seed)
    ids = [stoi[c] for c in start]
    for _ in range(n):
        x = torch.tensor([ids[-T:]])               # 直前8文字
        probs = F.softmax(model(x), dim=1)[0]
        ids.append(torch.multinomial(probs, 1, generator=g).item())
    return "".join(itos[i] for i in ids)

print(generate(model, "私はその人を常に"))